# 8.1. Deep Convolutional Neural Networks \(AlexNet\)

[AlexNet](https://en.wikipedia.org/wiki/AlexNet) was developed in 2012 and trained on the [ImageNet](https://www.image-net.org/) dataset with an accuracy that surpassed other computer vision models of its time. While it did not introduce revolutionary new ideas and architectures since LeNet in the early 2000s, it demonstrated that filters for complex features found in modern, high-resolution images ought to be learned automatically by the neural network and stacked through multiple convolutional layers rather than calculating and crafting them manually by hand, accelerating the key paradigm shift in the field of computer vision.

Python software and package versions used in this chapter are listed below powered by [GitCode Notebook](https://docs.gitcode.com/docs/help/home/ai-community/ai-notebook/).

1. Python 3.11
1. MindSpore 2.8.0
1. CANN 8.5.0

In [1]:
%pip install mindspore==2.8.0 \
    -i https://repo.mindspore.cn/pypi/simple \
    --trusted-host repo.mindspore.cn \
    --extra-index-url https://repo.huaweicloud.com/repository/pypi/simple

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://repo.mindspore.cn/pypi/simple, https://repo.huaweicloud.com/repository/pypi/simple

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


MindSpore version:  2.8.0


[WARNING] DEVICE(6500,fffeec6af120,python3.11):2026-05-09-15:43:41.042.188 [mindspore/ccsrc/plugin/ascend/res_manager/mem_manager/ascend_vmm_adapter.h:176] CheckVmmDriverVersion] Open file /etc/ascend_install.info failed.
[WARNING] DEVICE(6500,fffeec6af120,python3.11):2026-05-09-15:43:41.042.239 [mindspore/ccsrc/plugin/ascend/res_manager/mem_manager/ascend_vmm_adapter.h:204] CheckVmmDriverVersion] Open file /usr/local/Ascend/driver/version.info failed.


The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 8.1.2. AlexNet

AlexNet is essentially a modernized variation of LeNet with more layers. The layers are described below.

1. 1st convolutional layer with 96 output channels, an $11 \times 11$ kernel, a stride of 4 and a padding of `1px` in all directions followed by ReLU activation
1. 1st max pooling layer with a $3 \times 3$ kernel and a stride of 2
1. 2nd convolutional layer with 256 output channels, a $5 \times 5$ kernel, a stride of 1 and a padding of `2px` in all directions followed by ReLU activation
1. 2nd max pooling layer with a $3 \times 3$ kernel and a stride of 2
1. 3rd convolutional layer with 384 output channels, a $3 \times 3$ kernel, a stride of 1 and a padding of `1px` in all directions followed by ReLU activation
1. 4th convolutional layer with 384 output channels, a $3 \times 3$ kernel, a stride of 1 and a padding of `1px` in all directions followed by ReLU activation
1. 5th convolutional layer with 256 output channels, a $3 \times 3$ kernel, a stride of 1 and a padding of `1px` in all directions followed by ReLU activation
1. 3rd max pooling layer with a $3 \times 3$ kernel and a stride of 2
1. Flattening layer for the fully connected layers above
1. 1st fully connected layer with 4096 output channels followed by ReLU activation
1. 1st dropout layer with $p = 0.5$
1. 2nd fully connected layer with 4096 output channels followed by ReLU activation
1. 2nd dropout layer with $p = 0.5$
1. Final fully connected layer with 1000 output channels corresponding to the 1000 class labels in ImageNet

We'll modify AlexNet slightly to output 10 logits in the final layer corresponding to the Fashion MNIST dataset. Note that at the time AlexNet was developed and trained on ImageNet in 2012, modern, powerful deep learning libraries such as TensorFlow, PyTorch and MindSpore were yet to be invented. Furthermore, device memory constraints on NVIDIA GPUs of the time required manually splitting the model across 2 GPUs which was a remarkable feat of its time.

Today, we'll define AlexNet directly using high-level constructs provided by MindSpore and don't have to worry about splitting our model across 2 or more NPUs since our cloud-based Ascend 910B4 environment has 32G of available device memory which is sufficient for our use case.

In [3]:
!npu-smi info

+------------------------------------------------------------------------------------------------+
| npu-smi 25.5.1                   Version: 25.5.1                                               |
+---------------------------+---------------+----------------------------------------------------+
| NPU   Name                | Health        | Power(W)    Temp(C)           Hugepages-Usage(page)|
| Chip                      | Bus-Id        | AICore(%)   Memory-Usage(MB)  HBM-Usage(MB)        |
+===========================+===============+====================================================+
| 7     910B4               | OK            | 98.7        43                0    / 0             |
| 0                         | 0000:42:00.0  | 0           0    / 0          2946 / 32768         |
+===========================+===============+====================================================+
+---------------------------+---------------+----------------------------------------------------+
| NPU     

### 8.1.2.3. Capacity Control and Preprocessing

Here's our network defined with the high-level constructs provided by MindSpore.

In [4]:
import mindspore.nn as nn

alexnet = nn.SequentialCell([
    nn.Conv2d(1, 96, kernel_size=11, stride=4, pad_mode='pad', padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.Conv2d(96, 256, kernel_size=5),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.Conv2d(256, 384, kernel_size=3),
    nn.ReLU(),
    nn.Conv2d(384, 384, kernel_size=3),
    nn.ReLU(),
    nn.Conv2d(384, 256, kernel_size=3),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.Flatten(),
    nn.Dense(6400, 4096, activation='relu'),
    nn.Dropout(p=0.5),
    nn.Dense(4096, 4096, activation='relu'),
    nn.Dropout(p=0.5),
    nn.Dense(4096, 10)
])
alexnet

SequentialCell(
  (0): Conv2d(input_channels=1, output_channels=96, kernel_size=(11, 11), stride=(4, 4), pad_mode=pad, padding=1, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xfffeec6f9050>, bias_init=None, format=NCHW)
  (1): ReLU()
  (2): MaxPool2d(kernel_size=3, stride=2, pad_mode=VALID)
  (3): Conv2d(input_channels=96, output_channels=256, kernel_size=(5, 5), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xfffefb1f58d0>, bias_init=None, format=NCHW)
  (4): ReLU()
  (5): MaxPool2d(kernel_size=3, stride=2, pad_mode=VALID)
  (6): Conv2d(input_channels=256, output_channels=384, kernel_size=(3, 3), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xfffefb1a4850>, bias_init=None, format=NCHW)
  (7): ReLU()
  (8): Conv2d(input_channels=38

Of course, let's wrap it with MindSpore AMP so we don't have to manually cast our inputs to FP16 for matrix multiplication.

In [5]:
import mindspore.amp as amp

alexnet_amp = amp.auto_mixed_precision(network=alexnet, amp_level='O2')
alexnet_amp

_OutputTo32(
  (_backbone): SequentialCell(
    (0): Conv2d(input_channels=1, output_channels=96, kernel_size=(11, 11), stride=(4, 4), pad_mode=pad, padding=1, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xfffeec6f9050>, bias_init=None, format=NCHW)
    (1): ReLU()
    (2): MaxPool2d(kernel_size=3, stride=2, pad_mode=VALID)
    (3): Conv2d(input_channels=96, output_channels=256, kernel_size=(5, 5), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xfffefb1f58d0>, bias_init=None, format=NCHW)
    (4): ReLU()
    (5): MaxPool2d(kernel_size=3, stride=2, pad_mode=VALID)
    (6): Conv2d(input_channels=256, output_channels=384, kernel_size=(3, 3), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xfffefb1a4850>, bias_init=None, format=NCHW)
   

Recall the `layer_summary` function we defined in chapter 7.6 of D2L. Let's use it to inspect the output shape of each layer starting from a grayscale $224 \times 224$ image.

In [6]:
import mindspore.ops as ops

def layer_summary(net, X_shape):
    print(f'Input shape: {X_shape}')
    X = ops.randn(*X_shape)
    for cell in net.cells():
        X = cell(X)
        print(f'Output shape from {cell.__class__.__name__}: {X.shape}')

X_shape = (1, 1, 224, 224)
layer_summary(net=alexnet, X_shape=X_shape)

Input shape: (1, 1, 224, 224)
Output shape from Conv2d: (1, 96, 54, 54)
Output shape from ReLU: (1, 96, 54, 54)
Output shape from MaxPool2d: (1, 96, 26, 26)
Output shape from Conv2d: (1, 256, 26, 26)
Output shape from ReLU: (1, 256, 26, 26)
Output shape from MaxPool2d: (1, 256, 12, 12)
Output shape from Conv2d: (1, 384, 12, 12)
Output shape from ReLU: (1, 384, 12, 12)
Output shape from Conv2d: (1, 384, 12, 12)
Output shape from ReLU: (1, 384, 12, 12)
Output shape from Conv2d: (1, 256, 12, 12)
Output shape from ReLU: (1, 256, 12, 12)
Output shape from MaxPool2d: (1, 256, 5, 5)
Output shape from Flatten: (1, 6400)
Output shape from Dense: (1, 4096)
Output shape from Dropout: (1, 4096)
Output shape from Dense: (1, 4096)
Output shape from Dropout: (1, 4096)
Output shape from Dense: (1, 10)


## 8.1.3. Training

Let's train our variation of AlexNet on the Fashion MNIST dataset with the following hyperparameters.

1. Combined activation and loss function: softmax cross entropy with logits
1. Optimizer: minibatch SGD with a batch size of $2^7=128$
1. Learning rate: `0.01`
1. Epochs: at most 100 with early stopping, observe validation loss over 5 epochs and restore model weights from best epoch

We'll upsample our images from $28 \times 28$ to $224 \times 224$ resolution. This is suboptimal since we're performing more calculations based on the same available data but required to fit our Fashion MNIST thumbnail images to the input dimensions required by AlexNet.

The GitCode Notebook network environment is within Chinese mainland and unable to download the dataset from the upstream EU S3 bucket effectively. I have mirrored the dataset to my personal website [donaldsebleung.com](https://www.donaldsebleung.com/). It is hosted on Alibaba Cloud in Hong Kong region. Let's download it from there instead.

In [7]:
import os

dataset_dir = 'data/fashion/'
os.makedirs(dataset_dir, exist_ok=True)

In [8]:
import gzip
import urllib.request

prefix_url = 'https://donaldsebleung.com/assets/datasets/fashion-mnist'
X_train_url = f'{prefix_url}/train-images-idx3-ubyte.gz'
y_train_url = f'{prefix_url}/train-labels-idx1-ubyte.gz'
X_test_url = f'{prefix_url}/t10k-images-idx3-ubyte.gz'
y_test_url = f'{prefix_url}/t10k-labels-idx1-ubyte.gz'

X_train_path = os.path.join(dataset_dir, 'train-images-idx3-ubyte')
y_train_path = os.path.join(dataset_dir, 'train-labels-idx1-ubyte')
X_test_path = os.path.join(dataset_dir, 't10k-images-idx3-ubyte')
y_test_path = os.path.join(dataset_dir, 't10k-labels-idx1-ubyte')

with urllib.request.urlopen(X_train_url) as response:
    with open(X_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_train_url) as response:
    with open(y_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(X_test_url) as response:
    with open(X_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_test_url) as response:
    with open(y_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

In [9]:
import mindspore.dataset as ds

train_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='train', shuffle=True)
test_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='test', shuffle=True)

In [10]:
import mindspore.dataset.vision as vision
import mindspore.dataset.transforms as transforms
from mindspore import dtype as mstype

def transform_ds(dataset):
    image_transforms = [
        vision.Resize(size=(224, 224)),
        vision.Rescale(rescale=1/255, shift=0),
        vision.HWC2CHW()
    ]
    label_transforms = [
        transforms.OneHot(num_classes=10),
        transforms.TypeCast(data_type=mstype.float32)
    ]
    dataset = dataset.map(operations=image_transforms, input_columns='image')
    dataset = dataset.map(operations=label_transforms, input_columns='label')
    dataset = dataset.batch(batch_size=128, drop_remainder=False)
    return dataset

train_ds = transform_ds(dataset=train_ds)
test_ds = transform_ds(dataset=test_ds)

In [11]:
loss_fn = nn.SoftmaxCrossEntropyWithLogits(reduction='mean')
loss_fn

SoftmaxCrossEntropyWithLogits()

In [12]:
optimizer = nn.SGD(params=alexnet_amp.trainable_params(), learning_rate=0.01)
optimizer

SGD()

In [13]:
from mindspore.train import Model

model = Model(network=alexnet_amp, loss_fn=loss_fn, optimizer=optimizer, metrics={'accuracy', 'loss'})
model

In [14]:
from mindspore.train import EarlyStopping

early_stopping = EarlyStopping(patience=5, verbose=True, restore_best_weights=True)
early_stopping

In [15]:
epochs = 100

In [16]:
model.fit(epoch=epochs, train_dataset=train_ds, valid_dataset=test_ds, callbacks=[early_stopping])

.path string is NULLpath string is NULL..Restoring model weights from the end of the best epoch.
Epoch 00021: early stopping


Let's check the validation loss and accuracy of our trained AlexNet model against the Fashion MNIST dataset.

In [17]:
metrics = model.eval(valid_dataset=test_ds)
val_acc = metrics['accuracy']
val_loss = metrics['loss']
print(f'Validation loss: {val_loss:.4f}')
print(f'Validation accuracy: {val_acc:.4f}')

Validation loss: 0.3708
Validation accuracy: 0.8664


The validation accuracy is around $85\%$ - not bad!

## 8.1.4. Discussion

We saw how AlexNet from the 2012 ImageNet competition is simply a modernized, deep variant of LeNet from the early 2000s. Nevertheless, it obtained a classification accuracy far surpassing that of other computer vision models of its time and proved that complex features ought to be learned through multiple convolution layers instead of being designed by hand manually.